# Error handling

Every failure raises an exception from `atlas_local`, and the type says what
went wrong. This notebook triggers the common ones on purpose so you can see
what they look like before meeting them for real.

Requires a Docker daemon on the machine running this kernel.

In [ ]:
%pip install atlas-local-lib-py

## The exception hierarchy

All library-specific exceptions inherit from `AtlasLocalError`, so catching
`AtlasLocalError` handles errors raised by the library as a whole. Most
operations have a dedicated exception, while a few exceptions describe
conditions that can occur across multiple operations.

## A deployment that does not exist


In [1]:
from atlas_local import (
    AtlasLocalError,
    DeploymentTimeoutError,
    DockerConnectionError,
    GetConnectionStringError,
    GetDeploymentError,
    LocalDeployment,
    StartDeploymentError,
    UnhealthyDeploymentError,
)

try:
    deployment = LocalDeployment.get("unknown-deployment")
except GetDeploymentError as error:
    print(f"{type(error).__name__}: {error}")

GetDeploymentError: No local Atlas deployment found with that name or container ID.
Use `list()` to see the existing deployments.


## Conflicting options in `get_or_create`

`get_or_create` returns the existing deployment when one exists, but it does
not silently apply the requested options to it. If the requested options differ
from those of the existing deployment, it raises `ValueError` and lists the
fields whose values differ.

Only the options explicitly passed to `get_or_create` are compared, so calling
it without options always succeeds.


In [2]:
NAME = "error-handling-demo"

deployment = LocalDeployment.get_or_create(name=NAME)

try:
    LocalDeployment.get_or_create(name=NAME, image_tag="8.0", load_sample_data=True)
except ValueError as error:
    print(f"{type(error).__name__}: {error}")

ValueError: Deployment "error-handling-demo" already exists and does not match the requested options: image_tag, load_sample_data. Delete it first or omit the conflicting options.


## Operations that do not apply to the current state

Stopping and pausing are different lifecycle operations. Use `start()` after
`stop()`, and `unpause()` after `pause()`. Mixing these operations raises an
error.


In [3]:
deployment.pause()

try:
    deployment.start()
except StartDeploymentError as error:
    print(f"{type(error).__name__}: {error}")

deployment.unpause()

StartDeploymentError: Failed to start the deployment.
A paused deployment cannot be started; use `unpause()` or `stop()` first.


A stopped deployment is not running, so its connection string cannot be
retrieved until it is started again.


In [4]:
deployment.stop()

try:
    deployment.connection_string()
except GetConnectionStringError as error:
    print(f"{type(error).__name__}: {error}")

deployment.start()

GetConnectionStringError: The deployment does not have a published MongoDB port.
If it is stopped, call `start()` first; if it is paused, call `unpause()`.
If it was created without a published port, recreate it with the `port` argument to make it accessible from the host.


## Invalid arguments

Arguments are validated before Docker is contacted, so these fail immediately
and raise `ValueError` rather than a deployment error.

In [5]:
for description, call in [
    (
        "tag inside the image name",
        lambda: LocalDeployment.create(image="mongodb/mongodb-atlas-local:8.0"),
    ),
    (
        "negative timeout",
        lambda: LocalDeployment.start_deployment(NAME, wait_until_healthy_timeout=-1),
    ),
    ("negative tail", lambda: LocalDeployment.get_logs(NAME, tail=-1)),
]:
    try:
        call()
    except (ValueError, AtlasLocalError) as error:
        print(
            f"{description}\n  {type(error).__name__}: {str(error).splitlines()[0]}\n"
        )

tag inside the image name
  CreateDeploymentError: The image "mongodb/mongodb-atlas-local:8.0" must not include a tag.

negative timeout
  ValueError: wait_until_healthy_timeout must be non-negative number of seconds

negative tail
  ValueError: tail must be a non-negative number of lines



## Recoverable versus unexpected

Catch specific exceptions when you can take an appropriate action, and let unexpected exceptions propagate so they are not hidden. 
- `DeploymentTimeoutError` is worth retrying with a longer timeout.
- `UnhealthyDeploymentError` means the container did not become healthy and needs
looking at.
- `DockerConnectionError` is the user's environment, not your code.

Both `DeploymentTimeoutError` and `UnhealthyDeploymentError` inherit from
`WatchDeploymentError`, so catching that one covers "the deployment never
became usable" without distinguishing why.

In [6]:
def ensure_deployment(name):
    try:
        return LocalDeployment.get_or_create(name=name, wait_until_healthy_timeout=5)

    except DeploymentTimeoutError:
        print("Slow machine or cold image pull; waiting longer.")
        return LocalDeployment.get_or_create(name=name, wait_until_healthy_timeout=600)

    except UnhealthyDeploymentError as error:
        print(f"The deployment came up broken: {error}")
        print("Last log lines:")
        for line in LocalDeployment.get_logs(name, tail=5):
            print("   ", line.strip())
        raise

    except DockerConnectionError:
        print("Start Docker and run this cell again.")
        raise


deployment_2 = ensure_deployment("error-recovery-demo")
print("state:", deployment_2.state)
deployment_2.delete()
[other.name for other in LocalDeployment.list()]

Slow machine or cold image pull; waiting longer.
state: running


['error-handling-demo',
 'example-deployment-3',
 'example-deployment-2',
 'example-deployment-1']

## Clean up

In [ ]:
deployment.delete()
[other.name for other in LocalDeployment.list()]